# 03 – Clean

Apply the decisions from the data quality summary in `02_data_quality.ipynb`. Nothing is deleted: problem rows are flagged, not removed.

In [13]:
import pandas as pd

df = pd.read_csv("../data/processed/spend_combined.csv", dtype=str)
print(f"Loaded {len(df):,} rows")

# 1. Remove columns that are the same on every row
df = df.drop(columns=["council", "body_url"])

# 2. Trim stray spaces in text columns
for c in ["supplier", "expense_type", "service_area", "project"]:
    df[c] = df[c].fillna("").str.strip()

# 3. Amount: text -> number
df["amount"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False))

# 4. Dates: text -> date, plus the month of the file each row came from
df["pay_date"] = pd.to_datetime(df["pay_date"], format="%d/%m/%Y")
df["report_month"] = pd.to_datetime(
    df["source_file"].str.extract(r"spend_(\d{4}_\d{2})")[0], format="%Y_%m"
)
df["date_mismatch"] = df["pay_date"].dt.to_period("M") != df["report_month"].dt.to_period("M")

# 5. Blank project -> "No project"
df["project"] = df["project"].replace("", "No project")

# 6. Flags (rows are kept, only marked)
df["is_credit"] = df["amount"] < 0
df["is_zero"] = df["amount"] == 0
df["is_under_500"] = (df["amount"] > 0) & (df["amount"] < 500)

print(f"\nTotal: £{df['amount'].sum():,.2f}   (expect £1,597,049,909.20)")
print("Date mismatches:", df["date_mismatch"].sum(), "  (expect 99)")
print("Credits:", df["is_credit"].sum(), "  (expect 2,555)")
print("Zero-value:", df["is_zero"].sum(), "  (expect 23)")
print("Under £500:", df["is_under_500"].sum(), "  (expect 16,580)")
print('"No project":', (df["project"] == "No project").sum(), "  (expect 126,344)")
print()
print(df.dtypes.to_string())

Loaded 141,930 rows

Total: £1,597,049,909.20   (expect £1,597,049,909.20)
Date mismatches: 99   (expect 99)
Credits: 2555   (expect 2,555)
Zero-value: 23   (expect 23)
Under £500: 16580   (expect 16,580)
"No project": 126344   (expect 126,344)

supplier                         str
amount                       float64
pay_date              datetime64[us]
transaction_number               str
expense_type                     str
service_area                     str
project                          str
source_file                      str
report_month          datetime64[us]
date_mismatch                   bool
is_credit                       bool
is_zero                         bool
is_under_500                    bool


In [14]:
import re

# --- Supplier names ---
# Some months were exported with two garbled characters: "Æ" instead of ’ and "Ú" instead of é
df["supplier"] = df["supplier"].str.replace("Æ", "’", regex=False).str.replace("Ú", "é", regex=False)

def clean_name(s):
    s = s.upper().replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)       # remove punctuation
    s = re.sub(r"\bLIMITED\b", "LTD", s)    # Limited -> Ltd
    s = re.sub(r"\s+", " ", s).strip()      # tidy spaces
    return s

df["supplier_clean"] = df["supplier"].apply(clean_name)
print("Distinct supplier names:", df["supplier"].nunique(), "->", df["supplier_clean"].nunique())

# --- Spend groups ---
transfer_pattern = (
    "Pension Contributions|Enhanced Pension|Maintained Schools|Payments to Health Authorities|"
    "Grants|City Region Deal|Short Term Creditors|Loans & Other Financial|^BIDS$|^Internal "
)
is_transfer = (
    df["expense_type"].str.contains(transfer_pattern, case=False, regex=True)
    | df["service_area"].str.contains("Balance Sheet", case=False)
)
is_care = df["expense_type"].str.startswith("TPP")

df["spend_group"] = "Procurement"
df.loc[is_care, "spend_group"] = "Care and support"
df.loc[is_transfer, "spend_group"] = "Transfers and non-supplier"

# Payments to other public bodies (councils, NHS, ICBs, government departments) are not market purchases
is_public_body = (
    df["supplier_clean"].str.contains(r"\bCOUNCIL|\bNHS\b|DEPARTMENT FOR|\bICB\b", regex=True)
    & (df["spend_group"] == "Procurement")
)
df.loc[is_public_body, "spend_group"] = "Transfers and non-supplier"
print(f"Moved to Transfers (public bodies): {is_public_body.sum():,} payments, £{df.loc[is_public_body, 'amount'].sum() / 1e6:.1f}m")

summary = df.groupby("spend_group")["amount"].agg(payments="count", total="sum")
summary["share_%"] = (summary["total"] / summary["total"].sum() * 100).round(1)
summary["total"] = summary["total"].round(0)
print("\n", summary.to_string())

Distinct supplier names: 4452 -> 4428
Moved to Transfers (public bodies): 603 payments, £25.1m

                             payments        total  share_%
spend_group                                               
Care and support               49272  437394758.0     27.4
Procurement                    81449  808451670.0     50.6
Transfers and non-supplier     11209  351203481.0     22.0


In [15]:
# --- Reconciliation: our monthly sums vs the council's own total row ---
totals = pd.read_csv("../data/processed/file_totals.csv", dtype=str)
totals["official_total"] = pd.to_numeric(
    totals["amount"].str.replace(",", "", regex=False).str.replace("£", "", regex=False),
    errors="coerce",
)
mine = df.groupby("source_file")["amount"].sum().round(2).rename("our_total")

rec = totals.set_index("source_file")[["official_total"]].join(mine, how="outer")
rec["difference"] = (rec["our_total"] - rec["official_total"]).round(2)
rec["matches"] = rec["difference"].abs() < 1

print("Files that match the council's total (within £1):")
print(rec["matches"].value_counts().to_string())
print("\nFiles that do not match (or have no official total):")
print(rec[~rec["matches"]].to_string())

# --- Save the clean dataset ---
df.to_csv("../data/processed/spend_clean.csv", index=False)
print(f"\nSaved spend_clean.csv ({len(df):,} rows, {df.shape[1]} columns)")

Files that match the council's total (within £1):
matches
True     23
False     1

Files that do not match (or have no official total):
                   official_total    our_total  difference  matches
source_file                                                        
spend_2025_06.csv             NaN  58681599.05         NaN    False

Saved spend_clean.csv (141,930 rows, 15 columns)


## Cleaning summary

| Step | What was done |
|---|---|
| Columns | Dropped `council` and `body_url` (same value on every row). |
| Text | Trimmed stray spaces; fixed two garbled characters in supplier names (`Æ` → `’`, `Ú` → `é`). |
| Amounts | Removed commas and converted to numbers (credits stay negative). |
| Dates | Converted to real dates; added `report_month` from the file name. |
| Blanks | Blank `project` set to "No project". |
| Flags | `date_mismatch`, `is_credit`, `is_zero`, `is_under_500`. No rows were deleted. |
| Supplier names | Added `supplier_clean` (upper case, no punctuation, Limited → Ltd). |
| Spend groups | Added `spend_group`: Procurement, Care and support, Transfers and non-supplier. Supplier analysis uses Procurement only. Payments to other public bodies (councils, NHS, ICBs, government departments) are also moved to Transfers. |

## Reconciliation

The sum of cleaned amounts matches the council's own monthly total (within £1) for **23 of 24 files**. June 2025 has no total row in its file, so it could not be checked.